### **Problem Statement**

- Spark SQL is a Spark module for working with structured data
- Introduces an abstraction known as dataframes useful for working with SQL queries

- Dataset included with the course content.

### **Reference :**

- https://spark.apache.org/docs/latest/sql-programming-guide.html



### **Setting up Spark environment variables and installing dependencies**

In [103]:
import os # Importing the OS module

# Installing OpenJDK environment for running Spark
!apt-get install openjdk-11-jdk-headless -qq > /dev/null

# Downloading Apache Spark 3.5.1 with Hadoop quietly
!wget -q https://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz

# Extract the Spark Package
!tar xf spark-3.5.1-bin-hadoop3.tgz

# Set JAVA_HOME environment path pointing to where OpenJDK is installed
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

# Set SPARK_HOME environment path pointing to where Spark is installed
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"

In [104]:
# Installing the findspark package - necessary to run spark from Jupyter Notebook
!pip install findspark --quiet

# Importing the findspark library
import findspark

# Initializing PySpark - locates Spark installation
findspark.init()

### **A higher level and a slightly more straight forward approach for Spark installation**

In [71]:
!pip install pyspark

In [72]:
# Importing SparkSession from pyspark.sql, to create Spark entry points.

from pyspark.sql import SparkSession

In [73]:
# Calling the builder method and getOrCreate to create or
# retrieving a SparkSession
# spark runs locally with available cores


spark =  SparkSession.builder \
         .master("local[*]") \
         .appName("Spark SQL") \
         .getOrCreate()

In [74]:
spark

In [75]:
# Dependencies to import required files from Google Drive to Google Colab

from google.colab import drive

In [76]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [77]:
# Importing the dataset to Spark DataFrame

df = spark.read.csv("/content/drive/MyDrive/Indian_Cities_ZipCodes.csv", header = True, inferSchema = "true")

In [78]:
# Print the schema of the PySpark DataFrame in a tree format using .printSchema()
# method

df.printSchema()

root
 |-- RecordNumber: integer (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Zipcode: integer (nullable = true)
 |-- State: string (nullable = true)



In [79]:
# Shape of the dataframe

print("Shape of the dataframe is",(df.count(), len(df.columns)))

Shape of the dataframe is (33, 5)


In [80]:
# Display the first 20 rows of the PySpark DataFrame using df.show() method

df.show()

+------------+-------+----------+-------+-----+
|RecordNumber|Country|      City|Zipcode|State|
+------------+-------+----------+-------+-----+
|           1|  India|    Mumbai| 400001|   MH|
|           2|  India|      Pune| 411001|   MH|
|           3|  India|    Nagpur| 440001|   MH|
|           4|  India|   Lucknow| 226001|   UP|
|           5|  India|    Kanpur| 208001|   UP|
|           6|  India|      Agra| 282001|   UP|
|           7|  India| Bengaluru| 560001|   KA|
|           8|  India|    Mysore| 570001|   KA|
|           9|  India| Mangalore| 575001|   KA|
|          10|  India|   Chennai| 600001|   TN|
|          11|  India|Coimbatore| 641001|   TN|
|          12|  India|   Madurai| 625001|   TN|
|          13|  India|   Kolkata| 700001|   WB|
|          14|  India|   Asansol| 713301|   WB|
|          15|  India|  Durgapur| 713201|   WB|
|          16|  India| Hyderabad| 500001|   TG|
|          17|  India|  Warangal| 506002|   TG|
|          18|  India| Nizamabad| 503001

### **SELECTING MULTIPLE DATAFRAME COLUMNS**

In [81]:
# Using df.select() method in PySpark to select multiple dataframe columns
# displaying the first 5 rows

df.select("country","city","zipcode","state")\
  .show(5)

+-------+-------+-------+-----+
|country|   city|zipcode|state|
+-------+-------+-------+-----+
|  India| Mumbai| 400001|   MH|
|  India|   Pune| 411001|   MH|
|  India| Nagpur| 440001|   MH|
|  India|Lucknow| 226001|   UP|
|  India| Kanpur| 208001|   UP|
+-------+-------+-------+-----+
only showing top 5 rows



### **CREATING A TEMPORARY VIEW FOR AN SQL QUERY**

In [82]:
# Creating a temporary view called "Zipcodes" from the dataframe for SQL querying

df.createOrReplaceTempView("Zipcodes")

In [83]:
# SELECT Statement in Spark SQL to retrive the same column as df.select()
# displaying the first 5 rows

spark.sql("SELECT country, city, zipcode, state FROM Zipcodes") \
    .show(5)

+-------+-------+-------+-----+
|country|   city|zipcode|state|
+-------+-------+-------+-----+
|  India| Mumbai| 400001|   MH|
|  India|   Pune| 411001|   MH|
|  India| Nagpur| 440001|   MH|
|  India|Lucknow| 226001|   UP|
|  India| Kanpur| 208001|   UP|
+-------+-------+-------+-----+
only showing top 5 rows



### **FILTER ROWS USING THE  'WHERE' CONDITION**

In [84]:
# filter rows from the dataframe using select() and where() methods in PySpark
# display up to 10 rows using .show() method

df.select("country", "city", "zipcode", "state") \
  .where("state == 'TN'") \
  .show(10)

+-------+----------+-------+-----+
|country|      city|zipcode|state|
+-------+----------+-------+-----+
|  India|   Chennai| 600001|   TN|
|  India|Coimbatore| 641001|   TN|
|  India|   Madurai| 625001|   TN|
+-------+----------+-------+-----+



In [85]:
# Filtering the data with 'WHERE' statement in Spark SQL provides a response similar to
# df.select() and where() methods in PySpark

spark.sql(""" SELECT country, city, zipcode, state FROM Zipcodes
           WHERE state = 'TN'""").show(10)

+-------+----------+-------+-----+
|country|      city|zipcode|state|
+-------+----------+-------+-----+
|  India|   Chennai| 600001|   TN|
|  India|Coimbatore| 641001|   TN|
|  India|   Madurai| 625001|   TN|
+-------+----------+-------+-----+



### **GROUP AND AGGREGATING DATA**

In [86]:
# Using groupBy() and agg() to group by "State" column and
# calculate the count of unique zipcodes for each state

df.select("zipcode", "state") \
  .groupBy("state") \
  .agg({"zipcode" : "count"}) \
  .show()

+-----+--------------+
|state|count(zipcode)|
+-----+--------------+
|   BR|             3|
|   GJ|             3|
|   WB|             3|
|   TG|             3|
|   TN|             3|
|   UP|             3|
|   UK|             3|
|   MH|             3|
|   KA|             3|
|   RJ|             3|
|   MP|             3|
+-----+--------------+



In [87]:
# SQL Query within Spark, to count unique zipcodes
# for each state using 'GROUP BY' statement on zipcodes table

spark.sql(""" SELECT state, COUNT(zipcode) FROM zipcodes
GROUP BY state""").show()

+-----+--------------+
|state|count(zipcode)|
+-----+--------------+
|   BR|             3|
|   GJ|             3|
|   WB|             3|
|   TG|             3|
|   TN|             3|
|   UP|             3|
|   UK|             3|
|   MH|             3|
|   KA|             3|
|   RJ|             3|
|   MP|             3|
+-----+--------------+



### **SORT A COLUMN USING 'ORDER BY'**

In [88]:
# Filtering the dataframe using the where() method
# Sort the result by the 'state' column using OrderBy() method in PySpark

df.select("country", "city", "zipcode", "state") \
  .where("state in ('TN','TG','KA')")  \
  .orderBy("state") \
  .show(10)

+-------+----------+-------+-----+
|country|      city|zipcode|state|
+-------+----------+-------+-----+
|  India| Bengaluru| 560001|   KA|
|  India|    Mysore| 570001|   KA|
|  India| Mangalore| 575001|   KA|
|  India| Hyderabad| 500001|   TG|
|  India|  Warangal| 506002|   TG|
|  India| Nizamabad| 503001|   TG|
|  India|   Chennai| 600001|   TN|
|  India|Coimbatore| 641001|   TN|
|  India|   Madurai| 625001|   TN|
+-------+----------+-------+-----+



In [89]:
# Using WHERE and ORDER BY statements in Spark SQL to achieve the result
# similar to using df.where() and .orderBy() methods in PySpark
# Specific query selects and orders the records for the states 'TN', 'TG' and
# 'KA'

spark.sql(""" SELECT country, city, zipcode, state FROM
          Zipcodes WHERE state IN ('TN','TG','KA')
          ORDER BY state """).show(10)

+-------+----------+-------+-----+
|country|      city|zipcode|state|
+-------+----------+-------+-----+
|  India| Bengaluru| 560001|   KA|
|  India|    Mysore| 570001|   KA|
|  India| Mangalore| 575001|   KA|
|  India| Hyderabad| 500001|   TG|
|  India|  Warangal| 506002|   TG|
|  India| Nizamabad| 503001|   TG|
|  India|   Chennai| 600001|   TN|
|  India|Coimbatore| 641001|   TN|
|  India|   Madurai| 625001|   TN|
+-------+----------+-------+-----+



### **SPARK SQL 'JOIN' TYPES**

- Join in Spark works by combining data from two dataframes,  based on common column or key

- https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select-join.html

- The datasets required to work with Join in Spark SQL are provided


In [90]:
# Importing emp_dataset.csv into a spark dataframe

empDf = spark.read.option("header",True) \
        .csv("/content//drive/MyDrive/emp_dataset.csv")

In [91]:
# Display the schema of empDF dataframe

empDf.printSchema()

root
 |-- emp_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- superior_emp_id: string (nullable = true)
 |-- year_joined: string (nullable = true)
 |-- emp_dept_id: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: string (nullable = true)



In [92]:
# Shape of the dataframe

print("Shape of the dataframe is",(empDf.count(), len(empDf.columns)))

Shape of the dataframe is (1000, 7)


In [93]:
# Importing department_data.csv into a spark dataframe

deptDF = spark.read.option("header", True) \
        .csv("/content/drive/MyDrive/department_data.csv")

In [94]:
# Display the schema of deptDF dataframe

deptDF.printSchema()

root
 |-- dept_name: string (nullable = true)
 |-- dept_id: string (nullable = true)



In [95]:
# Display 10 rows of empDf DataFrame

empDf.show(10)

+------+------+---------------+-----------+-----------+------+------+
|emp_id|  name|superior_emp_id|year_joined|emp_dept_id|gender|salary|
+------+------+---------------+-----------+-----------+------+------+
|     1|Ishaan|             -1|       2011|         20|     F|  1503|
|     2| Ayaan|             -1|       2013|         10|     F|  4447|
|     3| Navya|              2|       2012|         40|     M|  1570|
|     4| Aarav|              2|       2012|         30|     M|  3799|
|     5|   Ira|              3|       2016|         40|     F|  2143|
|     6|Ananya|              5|       2017|         40|     F|  4234|
|     7|Vihaan|              2|       2018|         50|     F|  1927|
|     8| Aarav|              6|       2020|         40|     M|  4516|
|     9|Pranav|              3|       2018|         50|     M|  4107|
|    10| Advik|             -1|       2010|         20|     F|  3395|
+------+------+---------------+-----------+-----------+------+------+
only showing top 10 

In [96]:
# Display the rows of deptDF dataframe

deptDF.show()

+---------+-------+
|dept_name|dept_id|
+---------+-------+
|  Finance|     10|
|Marketing|     20|
|    Sales|     30|
|       IT|     40|
|       HR|     50|
+---------+-------+



#### **Inner Join**

- An Inner Join in Spark SQL returns the rows from the two dataframes
that have matching values in their specific columns provided as the join keys

In [97]:
# Performing an inner join in the two dataframes empDf and deptDF

# Inner join is based on the columns 'emp_dept_id' from empDf
# and 'dept_id' from deptDF provided as the join keys

# display the result using the .show() method without truncating
# the row information

empDf.join(deptDF, empDf.emp_dept_id == deptDF.dept_id, "inner") \
     .show(truncate = False)

+------+------+---------------+-----------+-----------+------+------+---------+-------+
|emp_id|name  |superior_emp_id|year_joined|emp_dept_id|gender|salary|dept_name|dept_id|
+------+------+---------------+-----------+-----------+------+------+---------+-------+
|1     |Ishaan|-1             |2011       |20         |F     |1503  |Marketing|20     |
|2     |Ayaan |-1             |2013       |10         |F     |4447  |Finance  |10     |
|3     |Navya |2              |2012       |40         |M     |1570  |IT       |40     |
|4     |Aarav |2              |2012       |30         |M     |3799  |Sales    |30     |
|5     |Ira   |3              |2016       |40         |F     |2143  |IT       |40     |
|6     |Ananya|5              |2017       |40         |F     |4234  |IT       |40     |
|7     |Vihaan|2              |2018       |50         |F     |1927  |HR       |50     |
|8     |Aarav |6              |2020       |40         |M     |4516  |IT       |40     |
|9     |Pranav|3              |2

#### **Left Outer Join**

- Returns all the rows from the left dataframe and the matching rows from the right dataframe

- If there are no matching values on the right dataframe, a null value is returned for the right dataframe's column.

In [98]:
# The left dataframe is empDF and the right dataframe is deptDF
# Left outer join is based on 'emp_dept_id' from empDf and
# 'dept_id' from deptDF provided as join keys

# Display 10 rows without truncation

empDf.join(deptDF, empDf.emp_dept_id == deptDF.dept_id, "leftouter") \
     .show(n = 10, truncate = False)

+------+------+---------------+-----------+-----------+------+------+---------+-------+
|emp_id|name  |superior_emp_id|year_joined|emp_dept_id|gender|salary|dept_name|dept_id|
+------+------+---------------+-----------+-----------+------+------+---------+-------+
|1     |Ishaan|-1             |2011       |20         |F     |1503  |Marketing|20     |
|2     |Ayaan |-1             |2013       |10         |F     |4447  |Finance  |10     |
|3     |Navya |2              |2012       |40         |M     |1570  |IT       |40     |
|4     |Aarav |2              |2012       |30         |M     |3799  |Sales    |30     |
|5     |Ira   |3              |2016       |40         |F     |2143  |IT       |40     |
|6     |Ananya|5              |2017       |40         |F     |4234  |IT       |40     |
|7     |Vihaan|2              |2018       |50         |F     |1927  |HR       |50     |
|8     |Aarav |6              |2020       |40         |M     |4516  |IT       |40     |
|9     |Pranav|3              |2

#### **Left-Anti Join**

- A Left Anti Join returns the rows from the left dataframe with no matching values in the right
dataframe

- If a row on the left dataframe has a matching value in the right dataframe, it would not be included in the result of the Left Anti Join

In [99]:
# The left dataframe is empDF and the right dataframe is deptDF
# left_anti join is based on 'emp_dept_id' column from empDf and
# 'dept_id' from deptDF provided as the join keys

# Displaying up to 10 rows without truncation

empDf.join(deptDF, empDf.emp_dept_id == deptDF.dept_id, "left_anti") \
     .show(n = 10, truncate = False)

+------+----+---------------+-----------+-----------+------+------+
|emp_id|name|superior_emp_id|year_joined|emp_dept_id|gender|salary|
+------+----+---------------+-----------+-----------+------+------+
+------+----+---------------+-----------+-----------+------+------+



#### **Right Outer Join**

- Right outer join returns all the rows from the right dataframe and the matching rows from the left dataframe

- if there no matching values in the left dataframe, a null value is returned corresponding to the left dataframe columns

In [100]:
# Right outer join returns all the rows from deptDF (right dataframe) and
# matching rows from empDF (left dataframe)

empDf.join(deptDF, empDf.emp_dept_id == deptDF.dept_id, "rightouter") \
     .show(n = 10, truncate = False)

+------+------+---------------+-----------+-----------+------+------+---------+-------+
|emp_id|name  |superior_emp_id|year_joined|emp_dept_id|gender|salary|dept_name|dept_id|
+------+------+---------------+-----------+-----------+------+------+---------+-------+
|1000  |Ritvik|295            |2017       |10         |M     |1147  |Finance  |10     |
|991   |Dhruv |944            |2014       |10         |M     |1817  |Finance  |10     |
|978   |Ayaan |480            |2013       |10         |F     |3595  |Finance  |10     |
|962   |Pranav|70             |2012       |10         |F     |2863  |Finance  |10     |
|935   |Anika |364            |2011       |10         |M     |2128  |Finance  |10     |
|926   |Arnav |601            |2016       |10         |M     |3750  |Finance  |10     |
|922   |Aanya |140            |2020       |10         |M     |1699  |Finance  |10     |
|916   |Ishaan|139            |2017       |10         |M     |4929  |Finance  |10     |
|905   |Vihaan|811            |2

#### **Full Outer Join**

A full outer join returns all the rows from both the left and right dataframe, including matching rows (where the join keys match) and non-matching rows from each dataframe

In [101]:
# Performing a full outer join between the left dataframe empDf and right dataframe deptDF
# 'emp_dept_id' from empDf and 'dept_id' from deptDF provided as the join keys

# Displaying 10 rows without truncating the content

empDf.join(deptDF, empDf.emp_dept_id == deptDF.dept_id, "full") \
    .show(n = 10, truncate = False)

+------+------+---------------+-----------+-----------+------+------+---------+-------+
|emp_id|name  |superior_emp_id|year_joined|emp_dept_id|gender|salary|dept_name|dept_id|
+------+------+---------------+-----------+-----------+------+------+---------+-------+
|2     |Ayaan |-1             |2013       |10         |F     |4447  |Finance  |10     |
|13    |Ishaan|12             |2012       |10         |F     |3882  |Finance  |10     |
|14    |Vivaan|6              |2013       |10         |F     |3046  |Finance  |10     |
|20    |Saanvi|10             |2020       |10         |M     |3806  |Finance  |10     |
|41    |Ritvik|12             |2013       |10         |M     |3723  |Finance  |10     |
|42    |Advik |25             |2020       |10         |M     |4023  |Finance  |10     |
|48    |Diya  |29             |2015       |10         |F     |1833  |Finance  |10     |
|49    |Dhruv |6              |2010       |10         |F     |4879  |Finance  |10     |
|55    |Navya |18             |2


- Semi-join on left dataframe 'empDf' and right dataframe 'deptDF'

- A semi-join returns rows from the left dataframe ('empDf') corresponding to a match with the right dataframe ('deptDF') based on the join key

In [102]:
# Performing a semi-join between the left dataframe 'empDf' and right dataframe 'deptDF'
# 'emp_dept_id' from empDf and 'dept_id' from deptDF provided as the join keys

empDf.join(deptDF, empDf.emp_dept_id == deptDF.dept_id, "leftsemi") \
    .show(n = 10, truncate = False)

+------+------+---------------+-----------+-----------+------+------+
|emp_id|name  |superior_emp_id|year_joined|emp_dept_id|gender|salary|
+------+------+---------------+-----------+-----------+------+------+
|1     |Ishaan|-1             |2011       |20         |F     |1503  |
|2     |Ayaan |-1             |2013       |10         |F     |4447  |
|3     |Navya |2              |2012       |40         |M     |1570  |
|4     |Aarav |2              |2012       |30         |M     |3799  |
|5     |Ira   |3              |2016       |40         |F     |2143  |
|6     |Ananya|5              |2017       |40         |F     |4234  |
|7     |Vihaan|2              |2018       |50         |F     |1927  |
|8     |Aarav |6              |2020       |40         |M     |4516  |
|9     |Pranav|3              |2018       |50         |M     |4107  |
|10    |Advik |-1             |2010       |20         |F     |3395  |
+------+------+---------------+-----------+-----------+------+------+
only showing top 10 